In [49]:
from pyspark.sql import functions as F

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 139, Finished, Available, Finished, False)

# **Customers**

In [50]:
customers_bronze = spark.read.table("bronze_customers")

# customers_bronze.printSchema()
# customers_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 140, Finished, Available, Finished, False)

In [51]:
gender_normalized = F.trim(F.lower(F.col("Gender"))) 
email_missing = F.col("Email").isNull() | F.lower(F.trim(F.col("Email"))).isin("", "-", "n/a", "null", "nan", "none")

def parse_my_date(date_col_name):
    return F.coalesce(
        F.to_date(F.col(date_col_name), "yyyy-MM-dd"),
        F.to_date(F.col(date_col_name), "dd/MM/yyyy"),
        F.to_date(F.col(date_col_name), "MM-dd-yyyy"),
        F.to_date(F.col(date_col_name), "dd MMM yyyy"),
        F.to_date(F.col(date_col_name), "yyyy/MM/dd"),
        F.to_date(F.col(date_col_name), "MMMM dd, yyyy")
    )

customers_cleaned = (
    customers_bronze
    .dropDuplicates(["CustomerID"])
    .withColumn("CustomerID", F.col("CustomerID").cast("bigint"))
    .withColumn("RegistrationDate", parse_my_date("RegistrationDate")) 
    .withColumn("Gender", 
                F.when(gender_normalized.isin("male","m","1"), "Male")
                .when(gender_normalized.isin("female","f","0"), "Female")
                .otherwise("unknown"))
    .withColumn("is_email_generated", F.when(email_missing, "Yes").otherwise("No"))
    .withColumn("Email", F.when(email_missing,
                F.lower(F.concat(F.col("FirstName"), F.lit("."), F.col("LastName"), F.lit("@example.com"))))
                .otherwise(F.col("Email")))
    )

# customers_cleaned.show()
# customers_cleaned.printSchema()

customers_cleaned.write.mode("overwrite").saveAsTable("silver_customers")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 141, Finished, Available, Finished, False)

# **Categories**

In [52]:
categories_bronze = spark.read.table("bronze_categories")
# categories_bronze.printSchema()
# categories_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 142, Finished, Available, Finished, False)

In [53]:
categories_cleaned = (
    categories_bronze
    .withColumn("CategoryID", F.col("CategoryID").cast("int"))
    .withColumn("MainCategory",
        F.when(F.col("CategoryName").isin("Clothing", "Shoes", "Jewelry"), "Fashion")
        .when(F.col("CategoryName").isin("Home & Kitchen", "Furniture", "Garden", "Tools"), "Home & Improvement")
        .when(F.col("CategoryName").isin("Electronics", "Office Supplies"), "Tech & Office")
        .when(F.col("CategoryName").isin("Groceries", "Health", "Beauty"), "Essentials & Wellness")
        .when(F.col("CategoryName").isin("Books", "Movies", "Music"), "Media")
        .when(F.col("CategoryName").isin("Baby Products", "Toys", "Pet Supplies", "Sports"), "Lifestyle & Family")
        .when(F.col("CategoryName").isin("Automotive"), "Automotive")
        .otherwise("Other"))
)

# categories_cleaned.printSchema()
# categories_cleaned.show()

categories_cleaned.write.mode("overwrite").saveAsTable("silver_categories")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 143, Finished, Available, Finished, False)

# **Departments**

In [54]:
departments_bronze = spark.read.table("Bronze_departments")

# departments_bronze.printSchema()
# departments_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 144, Finished, Available, Finished, False)

In [55]:
departments_cleaned = departments_bronze.withColumn("DepartmentID", F.col("DepartmentID").cast("int"))

# departments_cleaned.printSchema()

departments_cleaned.write.mode("overwrite").saveAsTable("silver_departments")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 145, Finished, Available, Finished, False)

# **Employees**

In [56]:
employees_bronze = spark.read.table("bronze_employees")

# employees_bronze.printSchema()
# employees_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 146, Finished, Available, Finished, False)

In [57]:
employees_cleaned = (
    employees_bronze
    .dropDuplicates()
    .withColumn("EmployeeID", F.col("EmployeeID").cast("int"))
    .withColumn("ManagerID", F.col("ManagerID").cast("int")) 
    .withColumn("DepartmentID", F.col("DepartmentID").cast("int"))
    .withColumn("Gender", F.when(gender_normalized.isin("male","m","1"), "Male")
                            .when(gender_normalized.isin("female","f","0"), "Female"))
    .withColumn("Salary", F.translate(F.col("Salary"), "$, ","").cast("decimal(18,2)"))
    .withColumn("HireDate", parse_my_date("HireDate"))
)

# employees_cleaned.printSchema()
# employees_cleaned.show()

employees_cleaned.write.mode("overwrite").saveAsTable("silver_employees")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 147, Finished, Available, Finished, False)

# **Order Details**

In [58]:
order_details_bronze = spark.read.table("bronze_order_details")

# order_details_bronze.printSchema()
# order_details_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 148, Finished, Available, Finished, False)

In [59]:
order_details_cleaned = (
    order_details_bronze
    .dropDuplicates()
    .withColumn("OrderDetailID", F.col("OrderDetailID").cast("bigint"))
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    .withColumn("ProductID", F.col("ProductID").cast("int"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("decimal(18,2)"))
    .withColumn("Discount", F.col("Discount").cast("float"))
    .withColumn("Discount",
        F.when(F.col("Discount") > 1, F.col("Discount") / 100)
        .otherwise(F.col("Discount")))
    .withColumn("Discount", F.col("Discount").cast("decimal(3,2)"))
    .withColumn("TotalSales", (F.col("UnitPrice") * F.col("Quantity") * (1 - F.col("Discount"))).cast("decimal(18,2)"))

)


# order_details_cleaned.printSchema()
# order_details_cleaned.show()

order_details_cleaned.write.mode("overwrite").saveAsTable("silver_order_details")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 149, Finished, Available, Finished, False)

# **Orders**

In [60]:
orders_bronze = spark.read.table("bronze_orders")

# orders_bronze.printSchema()
# orders_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 150, Finished, Available, Finished, False)

In [61]:
orders_cleaned = (
    orders_bronze
    .dropDuplicates()
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    .withColumn("CustomerID", F.col("CustomerID").cast("int")) 
    .withColumn("SalesEmployeeID", F.col("SalesEmployeeID").cast("int"))
    .withColumn("Status",F.trim(F.initcap(F.col("Status"))))
    .withColumn("Status",
                F.when(F.col("Status").isin("Cancelled", "Canceled"), "Cancelled")
                .otherwise(F.col("Status")))
    .withColumn("OrderDate", parse_my_date("OrderDate"))
)

# orders_cleaned.printSchema()
# orders_cleaned.show()

orders_cleaned.write.mode("overwrite").saveAsTable("silver_orders")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 151, Finished, Available, Finished, False)

# **Payments**

In [62]:
payments_bronze = spark.read.table("bronze_payments")

# payments_bronze.printSchema()
# payments_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 152, Finished, Available, Finished, False)

In [63]:
payments_cleaned = (
    payments_bronze
    .dropDuplicates()
    .withColumn("PaymentID", F.col("PaymentID").cast("bigint"))
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    .withColumn("Amount", F.translate(F.col("Amount"),"$, " ,"").cast("decimal(10,2)")) 
    .withColumn("PaymentStatus", F.trim(F.lower(F.col("PaymentStatus"))))
    .withColumn("PaymentStatus", 
            F.when(F.col("PaymentStatus").isin("done", "completed", "success"), "Completed")
            .when(F.col("PaymentStatus").isin("error", "declined", "failed"), "Failed")
            .otherwise(F.lit(None)))
    .withColumn("PaymentMethod", F.trim(F.lower(F.col("PaymentMethod"))))
    .withColumn("PaymentMethod", 
            F.when(F.col("PaymentMethod").isin("cod", "c.o.d", "cash" , "cash on delevry" , "cash on delivery"), "Cash on Delivery")
            .when(F.col("PaymentMethod").isin("mastercard" , "visa" , "credit card", "creditcard", "cc"), "Payment Card")
            .when(F.col("PaymentMethod").isNotNull(), F.initcap(F.col("PaymentMethod")))
            .otherwise(F.lit(None)))
    .withColumn("PaymentDate", parse_my_date("PaymentDate"))
)

# payments_cleaned.printSchema()
# payments_cleaned.show()

payments_cleaned.write.mode("overwrite").saveAsTable("silver_payments")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 153, Finished, Available, Finished, False)

# **Product Suppliers**

In [64]:
product_suppliers_bronze = spark.read.table("bronze_product_suppliers")

# product_suppliers_bronze.printSchema()
# product_suppliers_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 154, Finished, Available, Finished, False)

In [65]:
product_suppliers_cleaned = (
    product_suppliers_bronze
    .dropDuplicates()
    .withColumn("ProductID", F.col("ProductID").cast("int"))
    .withColumn("SupplierID", F.col("SupplierID").cast("int")) 
)

# product_suppliers_cleaned.printSchema()
# product_suppliers_cleaned.show()

product_suppliers_cleaned.write.mode("overwrite").saveAsTable("silver_product_suppliers")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 156, Finished, Available, Finished, False)

# **Products**

In [66]:
products_bronze = spark.read.table("bronze_products")

# products_bronze.printSchema()
# products_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 157, Finished, Available, Finished, False)

In [67]:
attributes_schema = "warranty_months INT, color STRING"

products_cleaned = (
    products_bronze
    .dropDuplicates(["ProductID"])
    .withColumn("ProductID", F.col("ProductID").cast("bigint"))
    .withColumn("CategoryID", F.col("CategoryID").cast("int"))
    .withColumn("Price", F.translate(F.col("Price"), "$, ", "").cast("decimal(10,2)"))
    .withColumn("Cost", F.translate(F.col("Cost"), "$, ", "").cast("decimal(10,2)"))
    .withColumn("Stock", F.col("Stock").cast("int"))
    .withColumn("StockType", 
                    F.when(F.col("Stock").isNull(), "Unknown") 
                    .when(F.col("Stock") < 0, "Invalid") 
                    .when(F.col("Stock") == 0, "No Stock") 
                    .when(F.col("Stock") <= 20, "Low Stock")                             
                    .when(F.col("Stock") <= 120, "normal Stock")                           
                    .otherwise("High Stock"))
    .withColumn("attr", F.from_json(F.regexp_replace(F.col("Attributes"), "'", '"'),  attributes_schema))
    .withColumn("warranty_months", F.col("attr.warranty_months"))
    .withColumn("color", F.col("attr.color"))
    .drop("Attributes", "attr")
)

# products_cleaned.printSchema()
# products_cleaned.show()

products_cleaned.write.mode("overwrite").saveAsTable("silver_products")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 158, Finished, Available, Finished, False)

# **Shipments**

In [68]:
shipments_bronze = spark.read.table("bronze_shipments")

# shipments_bronze.printSchema()
# shipments_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 159, Finished, Available, Finished, False)

In [69]:
shipments_cleaned = (
    shipments_bronze
    .dropDuplicates()
    .withColumn("ShipmentID", F.col("ShipmentID").cast("bigint"))
    .withColumn("OrderID", F.col("OrderID").cast("bigint"))
    .withColumn("ShipperID", F.col("ShipperID").cast("int"))
    .withColumn("ShipDate", parse_my_date("ShipDate"))
    .withColumn("DeliveryDate", parse_my_date("DeliveryDate"))
    .withColumn("DaysToArrive",F.datediff(F.col("DeliveryDate") , F.col("ShipDate")))
    .withColumn("DeliveryStatus", 
                F.when(F.col("DaysToArrive").isNull(), "Not Delivered")
                .when(F.col("DaysToArrive") < 0, "Invalid Dates")
                .otherwise("Delivered")
                )
)

# shipments_cleaned.printSchema()
# shipments_cleaned.show()

shipments_cleaned.write.mode("overwrite").saveAsTable("silver_shipments")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 160, Finished, Available, Finished, False)

# **Shippers**

In [70]:
shippers_bronze = spark.read.table("bronze_shippers")

# shippers_bronze.printSchema()
# shippers_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 161, Finished, Available, Finished, False)

In [71]:
shippers_cleaned = (
    shippers_bronze
    .dropDuplicates()
    .withColumn("ShipperID", F.col("ShipperID").cast("int"))
    .withColumn("ShipperType", 
            F.when(F.col("CompanyName").isin("USPS" , "Royal Mail" , "China Post"), "National Postal")
            .when(F.col("CompanyName").isin("DHL" , "FedEx" , "UPS" , "TNT"), "Global Express")
            .when(F.col("CompanyName").isin("Aramex" , "SF Express"), "Regional Specialists")
            .when(F.col("CompanyName").isin("Amazon Logistics"), "Captive E-commerce")
            .otherwise("Unknown")
            )
)

# shippers_cleaned.printSchema()
# shippers_cleaned.show()

shippers_cleaned.write.mode("overwrite").saveAsTable("silver_shippers")
# print("Done")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 162, Finished, Available, Finished, False)

# **Orders CDC Incremental**

In [72]:
orders_cdc_incremental_bronze = spark.read.table("bronze_orders_cdc_incremental")
# orders_cdc_incremental_bronze.printSchema()
# orders_cdc_incremental_bronze.show()

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 163, Finished, Available, Finished, False)

# **CDC**

In [73]:
from delta.tables import DeltaTable

source_table = spark.read.table("silver_orders_cdc_incremental")
target_table =  DeltaTable.forName(spark, "silver_orders")

(target_table.alias("target")
    .merge(source_table.alias("source"), "target.OrderID = source.OrderID")
    .whenMatchedUpdate(set = {"Status": "source.Status"})
    .execute())

# print("Merged")

StatementMeta(, be52c74f-3f9e-4489-944f-0b0b38c87f3e, 164, Finished, Available, Finished, False)